# 03 — Perplexity-based features (open LM) + stylometry -> LightGBM, with SHAP
**Kaggle settings:** Accelerator = GPU T4 x2 (one GPU is used), Internet = On.
**Inputs:** the competition data + the output of notebook 01 (`folds.csv`).
**Outputs:** `feats_ppl.parquet`, `val_ppl.csv`, `test_ppl.csv`, `submission_ppl.csv`, `figs/shap_*.png`.

Per token t we take from Qwen2.5-0.5B: log p(x_t), entropy H_t and the variance of log p under the
model's own distribution. Features: global statistics, Fast-DetectGPT-style curvature
(sum(logp + H) / sqrt(sum var), Bao et al., ICLR 2024, analytic version with the same model for
sampling and scoring), and **window-level** statistics (64-token windows). Fusion texts mix human and
AI passages, so we expect them to have a larger spread of window scores than pure human or pure AI.

In [ ]:
import os, glob, time, re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import GroupShuffleSplit
import lightgbm as lgb

INPUT_ROOT = os.environ.get("INPUT_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working"); os.makedirs(f"{OUT}/figs", exist_ok=True)
def find(name):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True))
    assert hits, f"{name} not found under {INPUT_ROOT}: add it via '+ Add Input'"
    return hits[0]

LM_NAME = "Qwen/Qwen2.5-0.5B"   # open-weight, multilingual (incl. German)
MAX_TOK, WIN, BS = 512, 64, 8
N_TRAIN = 30000                  # training-part rows used for the LightGBM model (fixed seed)
SEED = 42
LABELS = ["human", "ai", "fusion"]; L2I = {l: i for i, l in enumerate(LABELS)}

In [ ]:
train = pd.read_csv(find("train.csv")); test = pd.read_csv(find("test.csv"))
sample_sub = pd.read_csv(find("sample_submission.csv"))
work = pd.read_csv(find("folds.csv"))[["id", "fold", "group"]].merge(train, on="id", how="left")
tr = work[work.fold != 0]
tr = tr.groupby("label", group_keys=False).sample(frac=min(1.0, N_TRAIN / len(tr)), random_state=SEED)
va = work[work.fold == 0]
allx = pd.concat([tr.assign(part="train"), va.assign(part="val"), test.assign(part="test", fold=-1, group=-1)],
                 ignore_index=True)
print(allx.part.value_counts().to_dict())

In [ ]:
tok = AutoTokenizer.from_pretrained(LM_NAME)
lm = AutoModelForCausalLM.from_pretrained(LM_NAME, torch_dtype=torch.float16).cuda().eval()
if tok.pad_token is None: tok.pad_token = tok.eos_token

def win_stats(x, prefix):
    n = len(x) // WIN
    if n < 2:
        return {f"{prefix}_wstd": 0.0, f"{prefix}_wmin": float(x.mean()), f"{prefix}_wmax": float(x.mean()), f"{prefix}_wrange": 0.0}
    w = x[:n * WIN].reshape(n, WIN).mean(1)
    return {f"{prefix}_wstd": float(w.std()), f"{prefix}_wmin": float(w.min()), f"{prefix}_wmax": float(w.max()),
            f"{prefix}_wrange": float(w.max() - w.min())}

@torch.no_grad()
def lm_features(texts):
    enc = tok([t[:6000] for t in texts], truncation=True, max_length=MAX_TOK)["input_ids"]
    order = np.argsort([len(e) for e in enc]); feats = [None] * len(texts)
    for b in range(0, len(enc), BS):
        idx = order[b:b + BS]; batch = [enc[i] for i in idx]; L = max(map(len, batch))
        ii = torch.full((len(batch), L), tok.pad_token_id); am = torch.zeros((len(batch), L), dtype=torch.long)
        for j, s in enumerate(batch): ii[j, :len(s)] = torch.tensor(s); am[j, :len(s)] = 1
        logits = lm(input_ids=ii.cuda(), attention_mask=am.cuda()).logits
        for j, s in enumerate(batch):
            n = len(s) - 1
            if n < 2:
                feats[idx[j]] = {}; continue
            lp = torch.log_softmax(logits[j, :n].float(), -1)       # (n, V), per item to save memory
            tgt = torch.tensor(s[1:], device=lp.device)
            logp = lp.gather(1, tgt[:, None]).squeeze(1)
            p = lp.exp()
            H = -(p * lp).sum(-1)
            var = (p * lp.pow(2)).sum(-1) - H.pow(2)
            rank1 = (lp.argmax(-1) == tgt).float()
            logp, H, var, rank1 = [t.cpu().numpy() for t in (logp, H, var, rank1)]
            d = logp + H
            f = {"n_tok": n, "lp_mean": logp.mean(), "lp_std": logp.std(),
                 **{f"lp_q{q}": np.percentile(logp, q) for q in (5, 25, 50, 75)},
                 "ent_mean": H.mean(), "ent_std": H.std(), "top1_rate": rank1.mean(),
                 "fdgpt": d.sum() / np.sqrt(var.sum() + 1e-6), "d_mean": d.mean(), "d_std": d.std()}
            f.update(win_stats(logp, "lp")); f.update(win_stats(d, "d"))
            feats[idx[j]] = f
        del logits
    return pd.DataFrame(feats)

t0 = time.time()
F_lm = pd.concat([lm_features(allx.text.values[i:i + 4000].tolist()) for i in range(0, len(allx), 4000)], ignore_index=True)
print(f"LM features in {(time.time()-t0)/60:.1f} min", F_lm.shape)

In [ ]:
def stylo(texts):
    s = pd.Series(texts).fillna("")
    words = s.str.split(); n_ch = s.str.len().clip(lower=1)
    sent_len = s.str.split(r"(?<=[.!?])\s+").apply(lambda x: [len(t.split()) for t in x if t.strip()])
    f = pd.DataFrame({
        "n_words": words.str.len(), "n_chars": s.str.len(),
        "avg_word_len": words.apply(lambda w: np.mean([len(t) for t in w]) if w else 0),
        "avg_sent_len": sent_len.apply(lambda x: np.mean(x) if x else 0),
        "std_sent_len": sent_len.apply(lambda x: np.std(x) if x else 0),
        "ttr_300": words.apply(lambda w: len(set(w[:300])) / max(1, len(w[:300]))),
        "upper_rate": s.str.count(r"[A-ZÄÖÜ]") / n_ch, "digit_rate": s.str.count(r"\d") / n_ch,
    })
    for name, pat in {"comma": ",", "semi": ";", "colon": ":", "excl": "!", "q": r"\?", "dash": "[—–]",
                      "quote": "[\"“”„«»]", "paren": r"\(", "ellipsis": r"\.\.\.|…", "emo": r"[:;]-?[)(DP]"}.items():
        f[f"{name}_rate"] = s.str.count(pat) / n_ch * 1000
    return f

F = pd.concat([F_lm, stylo(allx.text.values)], axis=1)
F["lang"] = allx.lang.astype("category").cat.codes
F["domain"] = pd.Categorical(allx.domain).codes
F[["id", "part"]] = allx[["id", "part"]].values
F.to_parquet(f"{OUT}/feats_ppl.parquet", index=False)

# quick look: do the window-spread features separate fusion?
lab = allx.label
print(F[allx.part != "test"].groupby(lab[allx.part != "test"].values)[["lp_mean", "fdgpt", "lp_wstd", "d_wstd", "d_wrange"]]
      .median().loc[LABELS].round(3).to_string())

## LightGBM on LM + stylometric features
Early stopping uses a group-wise 10% inner split of the training part, not the validation fold.

In [ ]:
feat_cols = [c for c in F.columns if c not in ("id", "part")]
m_tr, m_va, m_te = (allx.part == "train").values, (allx.part == "val").values, (allx.part == "test").values
X_tr, y_tr = F.loc[m_tr, feat_cols].astype(float), allx.label[m_tr].map(L2I).values
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
i_fit, i_es = next(gss.split(X_tr, y_tr, allx.group[m_tr].values))
params = dict(objective="multiclass", num_class=3, learning_rate=0.05, num_leaves=63, min_child_samples=40,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=SEED)
dfit = lgb.Dataset(X_tr.iloc[i_fit], y_tr[i_fit], categorical_feature=["lang", "domain"])
des = lgb.Dataset(X_tr.iloc[i_es], y_tr[i_es], reference=dfit)
gbm = lgb.train(params, dfit, 3000, valid_sets=[des], callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])

X_va, y_va = F.loc[m_va, feat_cols].astype(float), allx.label[m_va].map(L2I).values
p_va = gbm.predict(X_va, num_iteration=gbm.best_iteration)
print("LM+stylo LightGBM val macro-F1:", round(f1_score(y_va, p_va.argmax(1), average="macro"), 4))
print(classification_report(y_va, p_va.argmax(1), target_names=LABELS, digits=4))

# ablation: which feature family carries the signal?
fam = {"stylometry only": [c for c in feat_cols if c in stylo(["a. b"]).columns or c in ("lang", "domain")],
       "LM global only": [c for c in feat_cols if c in F_lm.columns and "_w" not in c] + ["lang", "domain"],
       "LM global+window": list(F_lm.columns) + ["lang", "domain"]}
abl = []
for name, cols_ in fam.items():
    g = lgb.train(params, lgb.Dataset(X_tr.iloc[i_fit][cols_], y_tr[i_fit]), 3000,
                  valid_sets=[lgb.Dataset(X_tr.iloc[i_es][cols_], y_tr[i_es])], callbacks=[lgb.early_stopping(100, verbose=False)])
    abl.append({"features": name, "val_macro_f1": round(f1_score(y_va, g.predict(X_va[cols_]).argmax(1), average="macro"), 4)})
abl.append({"features": "all", "val_macro_f1": round(f1_score(y_va, p_va.argmax(1), average="macro"), 4)})
print(pd.DataFrame(abl).to_string(index=False)); pd.DataFrame(abl).to_csv(f"{OUT}/ppl_ablation.csv", index=False)

## SHAP: what the feature model relies on (xAI bonus)

In [ ]:
try:
    import shap
    Xs = X_va.sample(min(3000, len(X_va)), random_state=SEED)
    sv = shap.TreeExplainer(gbm).shap_values(Xs)
    sv = sv if isinstance(sv, list) else [sv[:, :, k] for k in range(3)]
    for k, l in enumerate(LABELS):
        plt.figure(); shap.summary_plot(sv[k], Xs, max_display=15, show=False)
        plt.title(f"SHAP — class {l}"); plt.tight_layout(); plt.savefig(f"{OUT}/figs/shap_{l}.png", dpi=110); plt.show()
except ImportError:
    print("shap not installed; using LightGBM gain importance")
imp = pd.Series(gbm.feature_importance("gain"), index=feat_cols).sort_values(ascending=False)
print(imp.head(15).round(0).to_string())

In [ ]:
p_te = gbm.predict(F.loc[m_te, feat_cols].astype(float), num_iteration=gbm.best_iteration)
cols = [f"s_{l}" for l in LABELS]
pd.DataFrame(np.log(p_va + 1e-9), columns=cols).assign(id=allx.id[m_va].values).to_csv(f"{OUT}/val_ppl.csv", index=False)
pd.DataFrame(np.log(p_te + 1e-9), columns=cols).assign(id=allx.id[m_te].values).to_csv(f"{OUT}/test_ppl.csv", index=False)
sub = sample_sub[["id"]].merge(pd.DataFrame({"id": allx.id[m_te].values, "label": np.array(LABELS)[p_te.argmax(1)]}), on="id")
assert len(sub) == len(sample_sub) and sub.label.isin(LABELS).all()
sub.to_csv(f"{OUT}/submission_ppl.csv", index=False)